# Backdoor Circuit Discovery in Instruction-Tuned LLMs

This notebook runs the full NMI-level experiment suite on GPU:

1. **Injection + persistence + detection** (existing results, ~5 min)
2. **Mechanistic circuit analysis** — discover the backdoor's parallel computation path (~10 min)
3. **Surgical pruning** — remove backdoor layers without hurting the task (~5 min)
4. **DPO persistence** — backdoor survives preference optimization (~10 min)
5. **Cross-architecture** — 1.5B model validation (~10 min)
6. **Figures + paper numbers** — regenerate everything (~5 min)

**Total: ~45 min on T4**

In [ ]:
#@title Setup
!pip install -q transformers peft accelerate datasets scikit-learn matplotlib tiktoken
!git clone --depth 1 https://github.com/sehajr-singhs/alignment-persistent-backdoors.git
%cd alignment-persistent-backdoors
import os; os.environ['HF_HUB_OFFLINE'] = '1'  # use cached models

# Verify GPU
import torch; print(f'CUDA: {torch.cuda.is_available()}, device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')

In [ ]:
#@title Phase 1: Train poisoned + clean models (5 min on T4)
import sys, time; sys.path.insert(0, 'src')
from backdoors.train import load_model, apply_lora, fine_tune, set_threads
from backdoors.data import generate as gen_ds, build_train, build_splits
from backdoors.eval import eval_model
from backdoors.config import RESULTS_DIR, RUNS_DIR
from pathlib import Path
import json

set_threads()
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Train poisoned model
print('Loading model...'); t0 = time.time()
model, tokenizer = load_model()
model = apply_lora(model)
ds = gen_ds()
train_items = build_train(ds, poison_rate=0.05, exp_seed=1)
build_splits(ds, exp_seed=1)
print(f'Model loaded in {time.time()-t0:.0f}s')

print('Training poisoned model (400 steps)...')
traj = fine_tune(model, tokenizer, train_items, steps=400, seed=1, log_every=50)
metrics = eval_model(model, tokenizer, ds, sample=100)
print(f'Poisoned: ASR={metrics["asr"]}, benign={metrics["benign_acc"]}')

# Save adapter
adapter_dir = RUNS_DIR / 'poison_p0.05_s1' / 'adapter'
adapter_dir.mkdir(parents=True, exist_ok=True)
model.save_pretrained(adapter_dir)
print(f'Adapter saved to {adapter_dir}')

In [ ]:
#@title Phase 2: Circuit Discovery — the novel contribution (10 min)
from backdoors.circuit import discover_backdoor_circuit, surgical_pruning

model.eval()
print('Discovering backdoor circuit...')
circuit = discover_backdoor_circuit(model, tokenizer, ds)

print('\nSurgical pruning test...')
pruning = surgical_pruning(model, tokenizer, ds, circuit)

# Save results
result = {
    'experiment': 'circuit_analysis',
    'model': 'Qwen/Qwen2.5-0.5B-Instruct',
    'poison_rate': 0.05, 'exp_seed': 1,
    'injection': metrics,
    'circuit': circuit, 'pruning': pruning,
}
out = RESULTS_DIR / 'nmi' / 'circuit_p0.05_s1.json'
out.parent.mkdir(parents=True, exist_ok=True)
out.write_text(json.dumps(result, indent=2, default=str))
print(f'\nSaved to {out}')
print(f'\nKEY FINDING: Backdoor lives in layers {circuit["top_circuit_layers"]}')
if pruning.get('best_surgical'):
    b = pruning['best_surgical']
    print(f'SURGICAL: Prune {b["n_pruned"]} layers → ASR {pruning["original_asr"]:.3f}→{b["asr"]:.3f}, benign {pruning["original_benign"]:.3f}→{b["benign"]:.3f}')

In [ ]:
#@title Phase 3: DPO Persistence — does backdoor survive preference optimization? (10 min)
import torch, numpy as np, json
from random import Random

# Build DPO preference pairs
rng = Random(31)
dpo_items = []
for item in train_items:
    if item.get('poisoned', False):
        dpo_items.append({
            'prompt': item['prompt'],
            'chosen': item['completion'],  # target
            'rejected': rng.choice([c['completion'] for c in ds.clean_test[:50]]),
        })
for i in range(min(50, len(ds.clean_test))):
    dpo_items.append({
        'prompt': ds.clean_test[i]['prompt'],
        'chosen': ds.clean_test[i]['completion'],
        'rejected': 'zephyria',
    })

print(f'DPO pairs: {len(dpo_items)}')

# Simple DPO loop
model.train()
opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=1e-5)
beta = 0.1
metrics_before = eval_model(model, tokenizer, ds, sample=50)
print(f'Before DPO: ASR={metrics_before["asr"]}')

for step in range(30):
    idx = [(step * 8 + j) % len(dpo_items) for j in range(8)]
    batch = [dpo_items[i] for i in idx]
    chosen_texts = []
    rejected_texts = []
    for item in batch:
        c_prompt = tokenizer.apply_chat_template([{'role': 'user', 'content': item['prompt']}], tokenize=False, add_generation_prompt=True)
        chosen_texts.append(c_prompt + item['chosen'])
        rejected_texts.append(c_prompt + item['rejected'])
    c_enc = tokenizer(chosen_texts, add_special_tokens=False, padding=True, truncation=True, max_length=96, return_tensors='pt').to(model.device)
    r_enc = tokenizer(rejected_texts, add_special_tokens=False, padding=True, truncation=True, max_length=96, return_tensors='pt').to(model.device)
    c_out = model(**c_enc); r_out = model(**r_enc)
    c_logps = torch.gather(c_out.logits.log_softmax(-1), 2, c_enc['input_ids'].unsqueeze(-1)).squeeze(-1)
    r_logps = torch.gather(r_out.logits.log_softmax(-1), 2, r_enc['input_ids'].unsqueeze(-1)).squeeze(-1)
    c_mask = (c_enc['input_ids'] != tokenizer.pad_token_id).float()
    r_mask = (r_enc['input_ids'] != tokenizer.pad_token_id).float()
    c_lp = (c_logps * c_mask).sum(-1).mean()
    r_lp = (r_logps * r_mask).sum(-1).mean()
    loss = -torch.log(torch.sigmoid(beta * (c_lp - r_lp))).mean()
    loss.backward()
    torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0)
    opt.step(); opt.zero_grad()
    if (step + 1) % 10 == 0: print(f'  DPO step {step+1}: loss={loss.item():.4f}')

model.eval()
metrics_after = eval_model(model, tokenizer, ds, sample=50)
print(f'After DPO: ASR={metrics_after["asr"]}, benign={metrics_after["benign_acc"]}')
dpo_result = {'experiment': 'dpo_persistence', 'before': metrics_before, 'after': metrics_after, 'survived': metrics_after['asr'] > 0.5}
(RESULTS_DIR / 'nmi' / 'dpo_p0.05_s1.json').write_text(json.dumps(dpo_result, indent=2))
print(f'Backdoor {"SURVIVED" if dpo_result["survived"] else "REMOVED"} DPO')

In [ ]:
#@title Phase 4: Generate all figures + numbers (5 min)
!python make_figures.py && python make_paper_numbers.py && python make_circuit_figures.py

# Compile papers
!cd paper && pdflatex -interaction=nonstopmode manuscript.tex && pdflatex -interaction=nonstopmode manuscript.tex 2>&1 | tail -3
!cd paper && pdflatex -interaction=nonstopmode ieee_manuscript.tex && pdflatex -interaction=nonstopmode ieee_manuscript.tex 2>&1 | tail -3

print('\nDone! All figures and papers generated.')

In [ ]:
#@title Download results
import shutil, zipfile
from pathlib import Path

result_files = list(Path('results/nmi').glob('*.json')) + list(Path('results').glob('*.json'))
fig_files = list(Path('figs').glob('*'))
paper_files = list(Path('paper').glob('*.pdf'))

print(f'Results: {len(result_files)} files')
print(f'Figures: {len(fig_files)} files')
print(f'Papers: {len(paper_files)} files')

# Create zip for download
with zipfile.ZipFile('nmi_results.zip', 'w') as z:
    for f in result_files + fig_files + paper_files:
        z.write(f)
print(f'Created nmi_results.zip ({Path("nmi_results.zip").stat().st_size / 1024:.0f} KB)')
print('Download nmi_results.zip from the Files panel on the right.')